In [5]:
!pip install transformers datasets evaluate accelerate sentencepiece huggingface_hub anthropic python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 33.6 MB/s eta 0:00:00


In [6]:
import json
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)

In [7]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")

CUDA available: True
GPU: Tesla T4


In [10]:
from google.colab import files

uploaded = files.upload()

Saving dataset.jsonl to dataset.jsonl


In [11]:
import os

print(os.listdir())

['.config', 'dataset.jsonl', 'sample_data']


In [12]:
import json
import pandas as pd

data = []

with open("dataset.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        data.append(json.loads(line))

df = pd.DataFrame(data)

print("Number of examples:", len(df))
df.head()

Number of examples: 200


,input,output
0,Let's schedule a quarterly business review eve...,"{'title': 'Quarterly Business Review', 'date':..."
1,Let's schedule a monthly budget review on the ...,"{'title': 'Budget Review', 'date': '2025-01-15..."
2,Let's schedule a quarterly business review eve...,"{'title': 'Quarterly Business Review', 'date':..."
3,Join us for the annual company picnic on July ...,"{'title': 'Annual Company Picnic', 'date': '20..."
4,Quarterly budget review with finance team ever...,"{'title': 'Quarterly Budget Review', 'date': '..."


In [13]:
import json

SCHEMA_INSTRUCTION = """
Convert the calendar request into valid compact JSON only.
Use exactly these fields:
title, date, time, duration_minutes, location, recurrence.
If a field is missing, use null.
Do not include explanation.
Calendar request:
"""

df["input_text"] = df["input"].apply(
    lambda x: SCHEMA_INSTRUCTION.strip() + " " + x
)

df["target_text"] = df["output"].apply(
    lambda x: json.dumps(x, sort_keys=True)
)

df[["input_text", "target_text"]].head()

,input_text,target_text
0,Convert the calendar request into valid compac...,"{""date"": ""2025-03-15"", ""duration_minutes"": 150..."
1,Convert the calendar request into valid compac...,"{""date"": ""2025-01-15"", ""duration_minutes"": 45,..."
2,Convert the calendar request into valid compac...,"{""date"": ""2024-04-01"", ""duration_minutes"": 150..."
3,Convert the calendar request into valid compac...,"{""date"": ""2025-07-19"", ""duration_minutes"": 180..."
4,Convert the calendar request into valid compac...,"{""date"": ""2025-04-05"", ""duration_minutes"": 120..."


In [14]:
from datasets import Dataset

dataset = Dataset.from_pandas(df[["input_text", "target_text"]])

dataset = dataset.train_test_split(test_size=0.2, seed=42)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 160
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 40
    })
})


In [15]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model loaded on:", device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded on: cuda


In [16]:
max_input_length = 256
max_target_length = 256

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=max_input_length,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=max_target_length,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["input_text", "target_text"]
)

tokenized_dataset

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 160
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 40
    })
})

In [17]:
from transformers import DataCollatorForSeq2Seq, TrainingArguments, Trainer

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

training_args = TrainingArguments(
    output_dir="./flan-t5-calendar-parser",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=15,
    weight_decay=0.01,
    logging_steps=10,
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,6.062468,4.856862
2,4.591648,3.532437
3,3.536513,2.581706
4,2.732347,1.876842
5,2.118834,1.445441
6,1.684767,1.174217
7,1.434313,0.984743
8,1.283298,0.855677
9,1.152284,0.757966
10,1.054636,0.696597


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=2.2085400994618736, metrics={'train_runtime': 838.3097, 'train_samples_per_second': 2.863, 'train_steps_per_second': 0.716, 'total_flos': 307108322906112.0, 'train_loss': 2.2085400994618736, 'epoch': 15.0})

In [22]:
def generate_calendar_json(text):
    prompt = SCHEMA_INSTRUCTION.strip() + " " + text

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [23]:
training_example = df.iloc[0]

print("Input:")
print(training_example["input"])

print("\nExpected Output:")
print(json.dumps(training_example["output"], sort_keys=True))

print("\nModel Output:")
print(generate_calendar_json(training_example["input"]))

Input:
Let's schedule a quarterly business review every three months starting March 15th, 2025 at 2pm in the large conference room on the 4th floor, lasting about two and a half hours

Expected Output:
{"date": "2025-03-15", "duration_minutes": 150, "location": "Large Conference Room, 4th Floor", "recurrence": "custom", "time": "14:00", "title": "Quarterly Business Review"}

Model Output:
"date": "2025-03-15", "duration_minutes": 90, "location": "Large Conference Room, 4th Floor", "recurrence": "custom", "time": "14:00", "title": "Quarterly Business Review"


In [24]:
test_examples = [
    "Meeting with Sarah on June 5th 2025 at 1pm for 30 minutes in the library",
    "Schedule a weekly team sync every Monday at 10 AM in Zoom",
    "Doctor appointment on March 12th 2025 at 3:30 PM for one hour",
    "Annual company picnic on July 19th 2025 at noon for 3 hours at Riverside Park"
]

for example in test_examples:
    print("Input:")
    print(example)
    print("Model Output:")
    print(generate_calendar_json(example))
    print("-" * 80)

Input:
Meeting with Sarah on June 5th 2025 at 1pm for 30 minutes in the library
Model Output:
"date": "2025-05-03-15", "duration_minutes": 90, "location": "Large Library", "recurrence": "recurrence": "recurrence": "recurrence": "recurrence": "recurrence": "recurrence
--------------------------------------------------------------------------------
Input:
Schedule a weekly team sync every Monday at 10 AM in Zoom
Model Output:
"date": "2025-01-10", "duration_minutes": 90, "location": "Large, "recurrence": "weekly", "time": "14:00", "title": "Team Sync"
--------------------------------------------------------------------------------
Input:
Doctor appointment on March 12th 2025 at 3:30 PM for one hour
Model Output:
"date": "2025-03-15", "duration_minutes": 90, "location": "Finance Conference Room", "recurrence": "recurrence": "recurrence": "recurrence": "recurrence": "recurrence": "recurrence
--------------------------------------------------------------------------------
Input:
Annual comp

In [25]:
model.save_pretrained("./flan-t5-calendar-parser")
tokenizer.save_pretrained("./flan-t5-calendar-parser")

print("Model and tokenizer saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved.


The initial Flan-T5 small model trained successfully and validation loss decreased, but generated outputs were not valid JSON. The model learned partial field names but failed to extract actual values, so we continued tuning with Flan-T5 base.

In [26]:
model.save_pretrained("./flan-t5-calendar-parser")
tokenizer.save_pretrained("./flan-t5-calendar-parser")

print("Model and tokenizer saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved.


In [27]:
from huggingface_hub import notebook_login

notebook_login()

In [28]:
repo_name = "flan-t5-calendar-parser-team10"

model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print("Model pushed to Hugging Face:", repo_name)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...1n0dxkw/model.safetensors:  10%|#         |  104MB /  990MB            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Model pushed to Hugging Face: flan-t5-calendar-parser-team10


In [2]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [29]:
from huggingface_hub import login
from getpass import getpass

hf_token = getpass("Paste your Hugging Face token: ")
login(token=hf_token)

Paste your Hugging Face token: ··········


In [18]:
repo_name = "flan-t5-calendar-parser-team10"

model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print("Model pushed to Hugging Face:", repo_name)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._vsbjha/model.safetensors:   0%|          |  553kB /  990MB            

README.md: 0.00B [00:00, ?B/s]

Model pushed to Hugging Face: flan-t5-calendar-parser-team10


## Final Summary

This notebook fine-tunes `google/flan-t5-small` on a 200-example synthetic calendar event dataset. The model converts natural language calendar requests into structured JSON fields: `title`, `date`, `time`, `duration_minutes`, `location`, and `recurrence`.

Training was completed in Google Colab using a Tesla T4 GPU for 15 epochs. Validation loss decreased from 6.94 to 2.35, showing improvement from the initial baseline.

Current limitation: generated outputs are not consistently valid JSON yet. This exported Hugging Face model should be treated as an initial fine-tuned baseline for backend integration testing, with further tuning needed before production use.